### 1. Importações e Configurações Iniciais

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)


### 2. Leitura e Preparação dos Dados

In [2]:
df = pd.read_csv("bank-full.csv", sep=";")

print("Dimensões do DataFrame:", df.shape)
display(df.head())


Dimensões do DataFrame: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [3]:
X = df.drop(columns="y")
y = (df["y"] == "yes").astype(int)

print("Dimensão de X:", X.shape)
print("\nDistribuição da variável-alvo y:")
print(y.value_counts(normalize=True).rename({0: "Não (0)", 1: "Sim (1)"}))


Dimensão de X: (45211, 16)

Distribuição da variável-alvo y:
y
Não (0)    0.883015
Sim (1)    0.116985
Name: proportion, dtype: float64


### 3. Divisão de Atributos Numéricos e Categóricos
Segundo a descrição das variáveis fornecida pela UCI.


In [4]:
numeric_features = [
    "age",
    "balance",
    "day",
    "duration",
    "campaign",
    "pdays",
    "previous"
]

categorical_features = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "poutcome"
]

print(f"Total de atributos numéricos: {len(numeric_features)}")
print(f"Total de atributos categóricos: {len(categorical_features)}")


Total de atributos numéricos: 7
Total de atributos categóricos: 9


### 4. Separação entre Treinamento e Teste

Separamos $20\%$ dos dados para o conjunto de teste final, utilizando amostragem estratificada (`stratify=y`) para preservar a proporção de classes desbalanceadas.


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Tamanho do Treino: {X_train.shape[0]} instâncias")
print(f"Tamanho do Teste:  {X_test.shape[0]} instâncias")


Tamanho do Treino: 36168 instâncias
Tamanho do Teste:  9043 instâncias


### 5. Definindo a Etapa de Pré-processamento

Utilizamos `ColumnTransformer` para padronizar atributos numéricos com `StandardScaler` e codificar variáveis categóricas com `OneHotEncoder`.


In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(), # z = (x - u) / s
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore", 
                sparse_output=False
            ),
            categorical_features
        )
    ]
)


### 6. Pipeline e GridSearch com KNN

In [7]:
# Definição do Pipeline KNN
pipeline_knn = Pipeline([
    ("preprocessing", preprocessor),
    ("model", KNeighborsClassifier())
])

# Estratégia de validação cruzada estratificada (3 folds)
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# Grade de hiperparâmetros para o KNN
param_grid_knn = {
    "model__n_neighbors": [3, 5, 7],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"]
}

grid_knn = GridSearchCV(
    estimator=pipeline_knn,
    param_grid=param_grid_knn,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    return_train_score=True
)

grid_knn.fit(X_train, y_train)

print("Melhores parâmetros KNN:")
print(grid_knn.best_params_)

print(f"\nMelhor F1-macro na Validação Cruzada: {grid_knn.best_score_:.4f}")


Melhores parâmetros KNN:
{'model__metric': 'euclidean', 'model__n_neighbors': 3, 'model__weights': 'uniform'}

Melhor F1-macro na Validação Cruzada: 0.6956


In [8]:
best_knn = grid_knn.best_estimator_
y_pred_knn = best_knn.predict(X_test)

print("Relatório de Classificação no Teste (KNN):")
print(classification_report(y_test, y_pred_knn))


Relatório de Classificação no Teste (KNN):
              precision    recall  f1-score   support

           0       0.92      0.96      0.94      7985
           1       0.58      0.36      0.45      1058

    accuracy                           0.89      9043
   macro avg       0.75      0.66      0.69      9043
weighted avg       0.88      0.89      0.88      9043



---
## EXERCÍCIOS RESOLVIDOS
---


### Questão 1 
Complemente o pipeline utilizado anteriormente para comparar diferentes classificadores no problema *bank-full*. Inclua no pipeline o **SVM** e o **Naive Bayes** com diferentes configurações de hiperparâmetros.


In [9]:
# 1. Pipeline e GridSearch para Naive Bayes (GaussianNB)
pipeline_nb = Pipeline([
    ("preprocessing", preprocessor),
    ("model", GaussianNB())
])

param_grid_nb = {
    "model__var_smoothing": [1e-9, 1e-7, 1e-5, 1e-3]
}

grid_nb = GridSearchCV(
    estimator=pipeline_nb,
    param_grid=param_grid_nb,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    return_train_score=True
)

grid_nb.fit(X_train, y_train)

print("=== NAIVE BAYES ===")
print("Melhores parâmetros:", grid_nb.best_params_)
print(f"Melhor F1-macro (CV): {grid_nb.best_score_:.4f}")

best_nb = grid_nb.best_estimator_
y_pred_nb = best_nb.predict(X_test)
print("\nRelatório de Teste (Naive Bayes):")
print(classification_report(y_test, y_pred_nb))


=== NAIVE BAYES ===
Melhores parâmetros: {'model__var_smoothing': 0.001}
Melhor F1-macro (CV): 0.6775

Relatório de Teste (Naive Bayes):
              precision    recall  f1-score   support

           0       0.93      0.90      0.92      7985
           1       0.41      0.53      0.46      1058

    accuracy                           0.85      9043
   macro avg       0.67      0.71      0.69      9043
weighted avg       0.87      0.85      0.86      9043



In [10]:
# 2. Pipeline e GridSearch para SVM (SVC)
pipeline_svm = Pipeline([
    ("preprocessing", preprocessor),
    ("model", SVC(random_state=42, cache_size=1000))
])

param_grid_svm = {
    "model__C": [0.1, 1.0],
    "model__kernel": ["linear", "rbf"]
}

grid_svm = GridSearchCV(
    estimator=pipeline_svm,
    param_grid=param_grid_svm,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    return_train_score=True
)

grid_svm.fit(X_train, y_train)

print("=== SVM (SVC) ===")
print("Melhores parâmetros:", grid_svm.best_params_)
print(f"Melhor F1-macro (CV): {grid_svm.best_score_:.4f}")

best_svm = grid_svm.best_estimator_
y_pred_svm = best_svm.predict(X_test)
print("\nRelatório de Teste (SVM):")
print(classification_report(y_test, y_pred_svm))


=== SVM (SVC) ===
Melhores parâmetros: {'model__C': 1.0, 'model__kernel': 'rbf'}
Melhor F1-macro (CV): 0.6948



Relatório de Teste (SVM):
              precision    recall  f1-score   support

           0       0.92      0.98      0.95      7985
           1       0.69      0.34      0.46      1058

    accuracy                           0.90      9043
   macro avg       0.80      0.66      0.70      9043
weighted avg       0.89      0.90      0.89      9043



Responda aqui.

1. Naive Bayes: foi adicionado com o estimador GaussianNB e variando var_smoothing.
2. SVM: foi adicionado com SVC variando C (0.1 e 1.0) e kernel (linear e rbf).

### Questão 2 
Explique como os três folds são utilizados para avaliar cada configuração dos algoritmos. Depois, para um dos algoritmos altere n_splits para 5 e compare os resultados obtidos com 3 folds. Discuta: o que mudou no experimento e quais são as possíveis vantagens e desvantagens de utilizar mais folds?


In [11]:
# Experimento comparativo com 5 Folds (utilizando Naive Bayes e KNN)
cv_5folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_nb_5folds = GridSearchCV(
    estimator=pipeline_nb,
    param_grid=param_grid_nb,
    cv=cv_5folds,
    scoring="f1_macro",
    n_jobs=-1,
    return_train_score=True
)

grid_nb_5folds.fit(X_train, y_train)

print(f"Naive Bayes - F1-macro (3 Folds): {grid_nb.best_score_:.4f}")
print(f"Naive Bayes - F1-macro (5 Folds): {grid_nb_5folds.best_score_:.4f}")

# Avaliação no teste
y_pred_nb_5 = grid_nb_5folds.best_estimator_.predict(X_test)
print(f"F1-macro no Teste (Modelo treinado com CV-5): {f1_score(y_test, y_pred_nb_5, average='macro'):.4f}")


Naive Bayes - F1-macro (3 Folds): 0.6775
Naive Bayes - F1-macro (5 Folds): 0.6795
F1-macro no Teste (Modelo treinado com CV-5): 0.6874


Responda aqui.

1. O conjunto de treino é dividido em 3 partes. Em cada rodada, 2 partes são usadas para treinar o pipeline e 1 parte para validar. O score final é a média dos 3 folds.
2. Com 5 folds, o modelo treina com 80% dos dados em cada rodada em vez de 66.7%, dando uma estimativa um pouco mais precisa.
3. Vantagens de mais folds: menor viés na avaliação e estimativa mais confiável.
4. Desvantagens: maior custo e tempo de processamento, pois treina mais vezes.

### Questão 3 
Quantas configurações diferentes serão testadas pelo GridSearchCV? Explique como o GridSearchCV utiliza o K-Fold para avaliar cada uma dessas combinações.


In [12]:
# Contagem exata das combinações testadas
configs_knn = len(param_grid_knn["model__n_neighbors"]) * len(param_grid_knn["model__weights"]) * len(param_grid_knn["model__metric"])
configs_nb = len(param_grid_nb["model__var_smoothing"])
configs_svm = len(param_grid_svm["model__C"]) * len(param_grid_svm["model__kernel"])

k_folds = 3

total_configs = configs_knn + configs_nb + configs_svm
total_fits = (configs_knn * k_folds) + (configs_nb * k_folds) + (configs_svm * k_folds)

df_contagem = pd.DataFrame([
    {"Algoritmo": "KNN", "Hiperparâmetros": "3 n_neighbors × 2 weights × 2 metric", "Configurações": configs_knn, "Folds": k_folds, "Total Treinamentos (Fits)": configs_knn * k_folds},
    {"Algoritmo": "Naive Bayes", "Hiperparâmetros": "4 var_smoothing", "Configurações": configs_nb, "Folds": k_folds, "Total Treinamentos (Fits)": configs_nb * k_folds},
    {"Algoritmo": "SVM (SVC)", "Hiperparâmetros": "2 C × 2 kernel", "Configurações": configs_svm, "Folds": k_folds, "Total Treinamentos (Fits)": configs_svm * k_folds},
    {"Algoritmo": "TOTAL", "Hiperparâmetros": "-", "Configurações": total_configs, "Folds": k_folds, "Total Treinamentos (Fits)": total_fits}
])

display(df_contagem)


,Algoritmo,Hiperparâmetros,Configurações,Folds,Total Treinamentos (Fits)
0,KNN,3 n_neighbors × 2 weights × 2 metric,12,3,36
1,Naive Bayes,4 var_smoothing,4,3,12
2,SVM (SVC),2 C × 2 kernel,4,3,12
3,TOTAL,-,20,3,60


Responda aqui.

1. Quantidade de configurações:
   - KNN: 12 configurações (3 n_neighbors x 2 weights x 2 metric)
   - Naive Bayes: 4 configurações (var_smoothing)
   - SVM: 4 configurações (2 C x 2 kernel)
   - Total: 20 configurações testadas (60 fits no total com 3 folds).
2. O GridSearchCV testa todas as combinações da grade, executando a validação cruzada para cada uma e escolhendo a que tiver a maior média da métrica.

### Questão 4
Após executar o GridSearchCV, compare grid.best_params_, grid.best_score_ de cada algoritmo e o desempenho obtido no conjunto de teste. Explique por que o melhor resultado da validação cruzada não precisa ser exatamente igual ao resultado obtido no teste.


In [13]:
# Comparação consolidada entre os modelos
modelos = [
    ("KNN", grid_knn, y_pred_knn),
    ("Naive Bayes", grid_nb, y_pred_nb),
    ("SVM (SVC)", grid_svm, y_pred_svm)
]

resumo_modelos = []
for nome, grid, y_pred in modelos:
    acc_test = accuracy_score(y_test, y_pred) * 100
    f1_test = f1_score(y_test, y_pred, average="macro")
    f1_class1 = f1_score(y_test, y_pred, pos_label=1)
    
    resumo_modelos.append({
        "Algoritmo": nome,
        "Melhores Hiperparâmetros": str(grid.best_params_).replace("model__", ""),
        "F1-macro (Validação CV)": f"{grid.best_score_:.4f}",
        "F1-macro (Teste)": f"{f1_test:.4f}",
        "Acurácia Teste (%)": f"{acc_test:.2f}%",
        "F1 Classe 'Sim' (1)": f"{f1_class1:.4f}"
    })

df_resumo = pd.DataFrame(resumo_modelos)
display(df_resumo)


,Algoritmo,Melhores Hiperparâmetros,F1-macro (Validação CV),F1-macro (Teste),Acurácia Teste (%),F1 Classe 'Sim' (1)
0,KNN,"{'metric': 'euclidean', 'n_neighbors': 3, 'wei...",0.6956,0.6948,89.47%,0.4478
1,Naive Bayes,{'var_smoothing': 0.001},0.6775,0.6874,85.47%,0.4588
2,SVM (SVC),"{'C': 1.0, 'kernel': 'rbf'}",0.6948,0.7028,90.49%,0.4578


Responda aqui.

1. Comparação:
   - KNN: F1-macro de 0.6956 na validação e 0.6948 no teste
   - Naive Bayes: F1-macro de 0.6775 na validação e 0.6874 no teste
   - SVM: F1-macro de 0.6948 na validação e 0.7028 no teste
2. Os resultados não são idênticos porque o modelo final é retreinado com 100% dos dados de treino, e o conjunto de teste é uma amostra separada e independente nunca vista pelo grid search.

### Questão 5 
Investigue os resultados de grid_knn.cv_results_ e identifique:
 - a configuração com maior F1-macro médio;
 - a configuração com maior F1-macro no treinamento;
 - a configuração com maior diferença entre treinamento e validação;
 - uma possível evidência de overfitting, caso exista.


In [14]:
# Análise da tabela de resultados da validação cruzada do KNN
df_cv_knn = pd.DataFrame(grid_knn.cv_results_)

# Cálculo da diferença entre treino e teste de validação
df_cv_knn["diff_train_val"] = df_cv_knn["mean_train_score"] - df_cv_knn["mean_test_score"]

# Seleção e formatação das colunas principais
colunas_exibicao = [
    "params",
    "mean_train_score",
    "mean_test_score",
    "std_test_score",
    "diff_train_val",
    "rank_test_score"
]

df_knn_analise = df_cv_knn[colunas_exibicao].sort_values(by="rank_test_score").reset_index(drop=True)
df_knn_analise["params"] = df_knn_analise["params"].apply(lambda d: {k.replace("model__", ""): v for k, v in d.items()})

display(df_knn_analise)


,params,mean_train_score,mean_test_score,std_test_score,diff_train_val,rank_test_score
0,"{'metric': 'euclidean', 'n_neighbors': 3, 'wei...",0.821404,0.695571,0.007069,0.125833,1
1,"{'metric': 'euclidean', 'n_neighbors': 3, 'wei...",1.000000,0.695465,0.006951,0.304535,2
2,"{'metric': 'euclidean', 'n_neighbors': 5, 'wei...",1.000000,0.687434,0.005974,0.312566,3
3,"{'metric': 'euclidean', 'n_neighbors': 5, 'wei...",0.769533,0.686926,0.006994,0.082606,4
4,"{'metric': 'manhattan', 'n_neighbors': 3, 'wei...",1.000000,0.684917,0.007063,0.315083,5
5,"{'metric': 'manhattan', 'n_neighbors': 3, 'wei...",0.815715,0.684765,0.007985,0.130950,6
6,"{'metric': 'euclidean', 'n_neighbors': 7, 'wei...",1.000000,0.682145,0.004389,0.317855,7
7,"{'metric': 'euclidean', 'n_neighbors': 7, 'wei...",0.743707,0.680446,0.003332,0.063261,8
8,"{'metric': 'manhattan', 'n_neighbors': 5, 'wei...",1.000000,0.677688,0.005230,0.322312,9
9,"{'metric': 'manhattan', 'n_neighbors': 5, 'wei...",0.759384,0.675410,0.005801,0.083974,10


In [15]:
# 1. Configuração com maior F1-macro médio na validação (Rank 1)
melhor_val = df_cv_knn.loc[df_cv_knn["rank_test_score"] == 1].iloc[0]

# 2. Configuração com maior F1-macro no treinamento
maior_train = df_cv_knn.loc[df_cv_knn["mean_train_score"].idxmax()]

# 3. Configuração com maior diferença entre treino e validação (Gap de Overfitting)
maior_gap = df_cv_knn.loc[df_cv_knn["diff_train_val"].idxmax()]

print(f"1. Maior F1-macro médio (Validação): {melhor_val['params']} -> Score: {melhor_val['mean_test_score']:.4f}")
print(f"2. Maior F1-macro no Treinamento:    {maior_train['params']} -> Score: {maior_train['mean_train_score']:.4f}")
print(f"3. Maior Diferença (Treino - Val):   {maior_gap['params']} -> Gap: {maior_gap['diff_train_val']:.4f} (Treino: {maior_gap['mean_train_score']:.4f} vs Val: {maior_gap['mean_test_score']:.4f})")


1. Maior F1-macro médio (Validação): {'model__metric': 'euclidean', 'model__n_neighbors': 3, 'model__weights': 'uniform'} -> Score: 0.6956
2. Maior F1-macro no Treinamento:    {'model__metric': 'euclidean', 'model__n_neighbors': 3, 'model__weights': 'distance'} -> Score: 1.0000
3. Maior Diferença (Treino - Val):   {'model__metric': 'manhattan', 'model__n_neighbors': 7, 'model__weights': 'distance'} -> Gap: 0.3320 (Treino: 1.0000 vs Val: 0.6680)


Responda aqui.

1. Maior F1-macro médio na validação: metric='euclidean', n_neighbors=3, weights='uniform' (score: 0.6956).
2. Maior F1-macro no treino: todas as configurações com weights='distance' (score: 1.0).
3. Maior diferença entre treino e validação: metric='manhattan', n_neighbors=7, weights='distance' (diferença de 0.3320).
4. Evidência de overfitting: sim, o uso de weights='distance' decora o treino (100%), mas tem desempenho pior na validação do que weights='uniform'.

# Conclusão

- O uso de Pipeline com ColumnTransformer evita vazamento de dados (data leakage).
- O SVM e o KNN tiveram os melhores resultados de F1-macro no teste.
- O Naive Bayes foi o mais rápido de treinar.
- Mais folds reduzem o viés da validação, mas aumentam o tempo de treino.
- No KNN, weights='distance' causou overfitting (100% no treino, pior na validação), enquanto weights='uniform' generalizou melhor.